# Schema Design

# Section A3 — Schema Justification

## Why This Schema Structure?

The database was modelled as a **Star Schema** with one fact table (`fact_events`) and three dimension tables (`dim_product`, `dim_user`, `dim_date`). This structure was chosen over a fully normalised 3NF model for a specific reason: all five analytical queries in Section C involve aggregating a measurable quantity (event count, revenue, user count) across one or more descriptive attributes (category, brand, hour, user). This is exactly the access pattern that star schemas are designed for.

In a fully normalised schema, answering "top 10 brands by revenue" would require joining `fact_events → dim_product → dim_brand → dim_category`. In the star schema, it requires a single join: `fact_events → dim_product`. At 110M rows, eliminating each additional join saves measurable seconds.

## Trade-offs Made

The `dim_product` table contains `category_main` and `category_sub` as derived columns rather than creating a separate `dim_category` table. This violates strict 3NF because these columns are functionally dependent on `category_code`, not on `product_key`. This denormalisation was chosen deliberately because the analytical queries access category as a property of each product lookup, not as an independent hierarchy. A separate `dim_category` table would add a join without enabling any additional analytical capability given the flat category structure in the source data.

## How Index Choices Support Section C Queries

`idx_events_type` eliminates full table scans in all five queries which filter on `event_type`. `idx_events_month` reduces scan scope from 110M to 55M rows for Q3 and Q4. `idx_user_key` makes the Q4 LEFT JOIN operate on an indexed column, delivering the largest speedup at 6.1x. `idx_events_session` converts the Q2 GROUP BY from a sort-based aggregation to an index scan.